# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. All entities are referenced by their `@id` as defined in the Croissant schema for reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is FAIR^2-compliant.

- Dataset URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We'll access the Croissant dataset using its schema URL. The metadata object lets us explore descriptive information in a structured form.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display metadata details
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, their IDs, and summarized field lists and `@id`s.

We'll enumerate the record sets and fields to orient further exploration. All record sets and fields are referenced using their `@id` for reproducibility.

In [ ]:
# List all record sets by @id and name
print("Available record sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  - @id: {rs['@id']} | name: {rs['name']}")

# For each record set, list the fields (columns)
print("\nFields for each record set (by @id):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']} | name: {rs['name']}")
    if 'fields' in rs and rs['fields']:
        for f in rs['fields']:
            print(f"    - Field @id: {f['@id']} | name: {f.get('name', '')}")
    else:
        print("    (no fields defined)")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the previous overview.

- We'll extract all available record sets shown above.
- For each, we show columns present and a sample of records.


In [ ]:
# Build a list of record set @id's for extraction
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nFirst few columns for RecordSet @id={rs_id}:")
    print(df.columns.tolist())
    print(df.head())

# For consistent reference in later sections, select the first record set for example EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nSelected record set for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Perform example data cleaning and transformations:

- Filter on a numeric field (e.g., peptides count or abundance; use field `@id`).
- Normalize the numeric field.
- Group data by a categorical field if present.

**Note:** Replace `<numeric_field_id>` and `<group_field_id>` with real field `@id`s as found in your previous overview for your EDA focus.

In [ ]:
# --- Example: EDA on peptides count or abundance ---
# Adjust these values after inspecting fields in section 2 & 3 for this dataset

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to infer a numeric field from column names
    sample_numeric_fields = [col for col in df.columns if ('count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower())]
    if sample_numeric_fields:
        numeric_field_id = sample_numeric_fields[0]
    else:
        numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns)>0 else df.columns[0]

    print(f"Using numeric field for analysis: {numeric_field_id}")
    # Filter by a threshold (e.g., abundance > 10), adapt threshold as necessary
    threshold = 10
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a suitable field (e.g., protein or type)
        group_fields = [col for col in df.columns if any(x in col.lower() for x in ['protein', 'type', 'category', 'sample'])]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print("No suitable categorical grouping field detected.")
    else:
        print(f"Field {numeric_field_id} is not numeric; please select a proper numeric field.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- We'll plot distributions of the main numeric field and (if possible) show a grouped average plot.

**Note:** You may need to further adapt this cell based on the dataset columns found in your session.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]
    # Plot distribution
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group_field was used above, plot mean by group
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        grouped_vals = df.groupby(group_field)[numeric_field_id].mean().sort_values()
        grouped_vals.plot(kind='bar')
        plt.title(f"Average {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 Croissant dataset for mass spectrometry analysis of extracellular vesicles from human mast cells. Key findings include:

- Records and fields are referenced by stable `@id`s for full reproducibility.
- The dataset supports advanced quantitative filtering and exploratory data analysis (EDA).
- The record sets include numeric measures suitable for normalization, filtering, and grouping.

You are now ready to extend this notebook to deeper domain analysis or machine learning workflows using the curated data!